# Extraction Runner

This notebook drives `Extraction.py` to produce hidden-state artifacts for every (model, dataset) pair in the registry.

    master_dataset.py  ->  datasets/<name>/processed/<name>_clean.csv
    Extraction.py      ->  hidden_states/<slug>/<dataset>/*.npy
    Probe.py           ->  interEx/<slug>/<dataset>/<trial>/*.csv

## How to read this notebook

Each code cell is preceded by a markdown cell that says what it does and why it comes at this position. Every code cell prints a **banner**, its **inputs**, its **work**, and its **outputs**, so a failure is reported where it happens.

| # | Cell | Purpose |
|---|---|---|
| 2 | Bootstrap  | imports, theme, path checks, system fingerprint |
| 4 | Datasets   | discover + validate processed CSVs |
| 6 | Models     | registry inventory, per-dataset completion |
| 8 | Pre-flight | RAM, cache, slug, manifest, disk checks |
| 10| Launch     | call `Extraction.run_experiments` |
| 12| Summary    | post-run matrix, ledger, timing chart |


In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — BOOTSTRAP
# ═══════════════════════════════════════════════════════════════════════

import importlib, json, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

display(HTML("""
<style>
    body, .jp-Notebook, .jp-OutputArea-output, .jp-RenderedHTMLCommon {
        background-color: #1e1e1e !important; color: #d4d4d4 !important;
    }
    h2, h3, h4 { color: #4fc3f7 !important; border-bottom: 2px solid #3498db !important; }
    b, strong { color: #f48fb1 !important; }
    .highlight { background-color: #2d2d2d !important; padding: 10px;
                 border-left: 4px solid #3498db; margin: 4px 0; color: #d4d4d4; }
    code { background-color: #333 !important; color: #ffcc80 !important;
           padding: 2px 4px; border-radius: 4px; }
    .dataframe { background-color: #2d2d2d !important; color: #d4d4d4 !important; }
</style>
"""))

def display_title(t): display(HTML(f"<h2>{t}</h2>"))
def display_info(m):  display(HTML(f"<div class='highlight'>{m}</div>"))

VERBOSITY = "quiet"
_ORDER = {"debug": 0, "info": 1, "quiet": 2}
def _say(msg, level="info"):
    if _ORDER[level] >= _ORDER[VERBOSITY]:
        print(msg)

def _banner(title):
    print(f"\n{'-' * 72}\n{title}\n{'-' * 72}")

import _shared
importlib.reload(_shared)
import Extraction as EX
importlib.reload(EX)

from _shared import (
    AMIRALI_MOUNT, DATASETS_ROOT, MODELS_ROOT,
    HIDDEN_STATES_ROOT, INTEREX_ROOT,
    HF_HUB_CACHE, EXTRACTION_META_ROOT,
    model_slug,
)

_banner("STEP 1 / 5 — Path verification")

REQUIRED = {"Drive mount": AMIRALI_MOUNT, "Datasets root": DATASETS_ROOT}
OPTIONAL = {
    "Models root":     MODELS_ROOT,
    "Hidden states":   HIDDEN_STATES_ROOT,
    "InterEx outputs": INTEREX_ROOT,
    "HF hub cache":    HF_HUB_CACHE,
    "Extraction meta": EXTRACTION_META_ROOT,
}
for label, p in REQUIRED.items():
    assert p.is_dir(), f"Required path missing: {p}"
    _say(f"  [OK]  {label:18s}  {p}")
for label, p in OPTIONAL.items():
    mark = "[OK]" if p.is_dir() else "[--]"
    note = "" if p.is_dir() else "  (created on first run)"
    _say(f"  {mark}  {label:18s}  {p}{note}")

_banner("STEP 1 / 5 — System")

import platform, torch, transformers
_say(f"  Platform       : {platform.platform()}")
_say(f"  Python         : {platform.python_version()}")
_say(f"  PyTorch        : {torch.__version__}")
_say(f"  Transformers   : {transformers.__version__}")
_say(f"  CPU threads    : {torch.get_num_threads()}")
try:
    import psutil
    vm = psutil.virtual_memory()
    _say(f"  RAM total      : {vm.total / 1e9:.2f} GB")
    _say(f"  RAM available  : {vm.available / 1e9:.2f} GB")
    _say(f"  RAM used       : {vm.percent}%")
except ImportError:
    _say("  RAM info       : psutil not installed (pip install psutil)")
_say(f"  Device         : {'cuda' if torch.cuda.is_available() else 'cpu'}")
print("\nOK Bootstrap complete. Next cell discovers processed datasets.")



------------------------------------------------------------------------
STEP 1 / 5 — Path verification
------------------------------------------------------------------------
  [OK]  Drive mount         /Volumes/Amirali
  [OK]  Datasets root       /Volumes/Amirali/datasets
  [OK]  Models root         /Volumes/Amirali/models
  [OK]  Hidden states       /Volumes/Amirali/hidden_states
  [OK]  InterEx outputs     /Volumes/Amirali/interEx
  [OK]  HF hub cache        /Volumes/Amirali/.hf_cache/hub
  [OK]  Extraction meta     /Volumes/Amirali/hidden_states/_meta

------------------------------------------------------------------------
STEP 1 / 5 — System
------------------------------------------------------------------------
  Platform       : macOS-14.6.1-x86_64-i386-64bit
  Python         : 3.12.0
  PyTorch        : 2.2.2
  Transformers   : 4.57.6
  CPU threads    : 4
  RAM total      : 8.59 GB
  RAM available  : 3.68 GB
  RAM used       : 57.1%
  Device         : cpu

OK Bootstrap comp

## Processed datasets

`master_dataset.py` writes one CSV per dataset at `datasets/<name>/processed/<name>_clean.csv`. `EX.discover_processed_datasets` walks that root and returns `DATASETS` (`{name: DataFrame}`) and `CSV_HASHES` (`{name: sha256}`). Both are needed later: the DataFrames are tokenized during extraction, and the hashes are recorded in every artifact so a probe run can prove it is looking at the same CSV.

### Contract enforced by the next cell

| Check | Why |
|---|---|
| Columns are exactly `clean_text, label, sentiment_score` | Extraction expects them |
| Index is a clean `RangeIndex 0..N-1` | Row `i` in CSV maps to row `i` in `hidden_states.npy` |
| Every label is a non-empty Python list | The probe's label adapter assumes it |
| No null `clean_text` rows | Extraction would drop them and desync |

The next cell raises `AssertionError` if any dataset violates the contract. A bad dataset must not reach the extraction stage.


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — DATASET DISCOVERY & CONTRACT CHECK
# ═══════════════════════════════════════════════════════════════════════

_banner("STEP 2 / 5 — Processed dataset discovery")

DATASETS, CSV_HASHES = EX.discover_processed_datasets(show_info=True)
assert DATASETS, "No processed datasets discovered."

_failures = []
for name, df in DATASETS.items():
    if list(df.columns) != list(EX.PROCESSED_COLUMNS):
        _failures.append(f"{name}: bad columns {list(df.columns)}")
        continue
    if not (df.index.is_unique and df.index[0] == 0
            and df.index[-1] == len(df) - 1):
        _failures.append(f"{name}: index is not a clean RangeIndex")
    if not df["label"].map(lambda x: isinstance(x, list) and len(x) > 0).all():
        _failures.append(f"{name}: some labels are not non-empty lists")
    nulls = int(df["clean_text"].isna().sum())
    if nulls:
        _failures.append(f"{name}: {nulls} null clean_text rows")

if _failures:
    print()
    for f in _failures:
        print(f"  X {f}")
    raise AssertionError(f"{len(_failures)} dataset(s) failed the contract")

_banner("STEP 2 / 5 — Discovery summary")
summary = pd.DataFrame([
    {
        "name":         name,
        "rows":         f"{len(df):,}",
        "n_classes":    str(len(set(x for row in df["label"] for x in row))),
        "sample_label": str(df["label"].iloc[0]),
        "csv_sha256":   CSV_HASHES[name][:16] + "...",
    }
    for name, df in DATASETS.items()
])
display(summary)
display_info(f"<b>{len(DATASETS)}</b> dataset(s) satisfy the contract. Next cell enumerates the model registry.")



------------------------------------------------------------------------
STEP 2 / 5 — Processed dataset discovery
------------------------------------------------------------------------
  ✓ amazon_polarity            3,993,715 rows  sha256=869cdc40c857  (amazon_polarity/processed/amazon_polarity_clean.csv)
  ✓ emotion                      19,999 rows  sha256=5306dde7d824  (emotion/processed/emotion_clean.csv)
  ✓ goemo                        54,039 rows  sha256=567885691ac5  (goemo/processed/goemo_clean.csv)
  ✓ isear                         7,532 rows  sha256=075d2d987106  (isear/processed/isear_clean.csv)
  ✓ sst2                         67,855 rows  sha256=fadcbf89bc37  (sst2/processed/sst2_clean.csv)
  ✓ tweet_eval_emotion            5,027 rows  sha256=2791d0b2c71d  (tweet_eval_emotion/processed/tweet_eval_emotion_clean.csv)

  · emotion_dataset_100k       skipped — no processed CSV
  · empathetic                 skipped — no processed CSV

---------------------------------------

,name,rows,n_classes,sample_label,csv_sha256
0,amazon_polarity,"3,993,715",2,[1],869cdc40c85740ca...
1,emotion,"19,999",6,[0],5306dde7d8241c52...
2,goemo,"54,039",28,[27],567885691ac5f48b...
3,isear,"7,532",7,[1],075d2d9871063d19...
4,sst2,"67,855",2,[0],fadcbf89bc3729a1...
5,tweet_eval_emotion,"5,027",4,[2],2791d0b2c71d6d26...


## Model registry & completion

`Extraction.MODEL_REGISTRY` lists all 25 models the pipeline knows about. For each we ask: is a hidden-state artifact already present for every dataset, and does the model fit in this machine's RAM at fp32?

### Two hard skip sets

`GATED` — models requiring Hugging Face authentication. Loading one raises `GatedRepoError`. Every Gemma-3 and Llama-3.2 variant.

`TOO_BIG` — models whose fp32 load peak exceeds ~7 GB. On an 8 GB machine they OOM during load. Extract them on a larger machine and copy the artifacts back.

### `TARGET_MODELS`

The next cell computes this list: every non-gated, non-oversized model that is still missing at least one dataset. The launch cell iterates over exactly this list. If it is empty, nothing runs.


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — MODEL INVENTORY
# ═══════════════════════════════════════════════════════════════════════

_banner("STEP 3 / 5 — Model registry inventory")
EXCLUDE_DATASETS = {"amazon_polarity"}

DATASETS_FILTERED    = {k: v for k, v in DATASETS.items()    if k not in EXCLUDE_DATASETS}
CSV_HASHES_FILTERED  = {k: v for k, v in CSV_HASHES.items()  if k not in EXCLUDE_DATASETS}
DS_NAMES             = list(DATASETS_FILTERED.keys())

GATED = {
    "google/gemma-3-270m",   "google/gemma-3-1b-pt",  "google/gemma-3-4b-pt",
    "meta-llama/Llama-3.2-1B", "meta-llama/Llama-3.2-3B",
}
TOO_BIG = {
    "Qwen/Qwen2.5-3B", "Qwen/Qwen3-1.7B-Base", "Qwen/Qwen3-4B-Base",
}

def dataset_is_done(model, dataset):
    """True iff <slug>/<dataset>/completed.npy exists and is all-True."""
    d = EX.build_dataset_directory(model, dataset)
    comp, meta = d / "completed.npy", d / "extraction.json"
    if not (comp.is_file() and meta.is_file()):
        return False
    try:
        return bool(np.load(comp, mmap_mode="r").all())
    except Exception:
        return False

ALL_MODELS = list(EX.ALL_PRIMARY_MODEL_NAMES)
DS_NAMES   = list(DATASETS.keys())


matrix = pd.DataFrame(
    [[("V" if dataset_is_done(m, d) else ".") for d in DS_NAMES]
     for m in ALL_MODELS],
    index=[m.split("/")[-1] for m in ALL_MODELS],
    columns=DS_NAMES,
)
matrix["done"] = (matrix[DS_NAMES] == "V").sum(axis=1).astype(str) + f"/{len(DS_NAMES)}"
matrix["skip"] = [
    "gated"   if m in GATED   else
    "too_big" if m in TOO_BIG else
    ""
    for m in ALL_MODELS
]
display(matrix)

TARGET_MODELS = [
    m for m in ALL_MODELS
    if m not in GATED | TOO_BIG
    and not all(dataset_is_done(m, d) for d in DS_NAMES)
]

_banner("STEP 3 / 5 — Target list for this run")
_say(f"  Registry total   : {len(ALL_MODELS)}")
_say(f"  Gated / too big  : {len(GATED | TOO_BIG)}")
_say(f"  Target this run  : {len(TARGET_MODELS)}")
for m in TARGET_MODELS:
    n = sum(dataset_is_done(m, d) for d in DS_NAMES)
    _say(f"    - {m:<55}  {n}/{len(DS_NAMES)} done")

if not TARGET_MODELS:
    display_info("Nothing to extract — every non-gated model has all datasets.")
else:
    display_info(f"<b>{len(TARGET_MODELS)}</b> model(s) will be extracted. Next cell runs pre-flight checks.")



------------------------------------------------------------------------
STEP 3 / 5 — Model registry inventory
------------------------------------------------------------------------


,amazon_polarity,emotion,goemo,isear,sst2,tweet_eval_emotion,done,skip
bert-base-uncased,.,V,V,V,V,V,5/6,
distilbert-base-uncased,.,.,.,.,.,.,0/6,
roberta-base,.,.,.,.,.,.,0/6,
electra-small-discriminator,.,.,.,.,.,.,0/6,
deberta-v3-small,.,.,.,.,.,.,0/6,
gpt2,.,.,.,.,.,.,0/6,
gpt-neo-125m,.,.,.,.,.,.,0/6,
opt-125m,.,.,.,.,.,.,0/6,
SmolLM2-135M,.,.,.,.,.,.,0/6,
SmolLM2-360M,.,.,.,.,.,.,0/6,



------------------------------------------------------------------------
STEP 3 / 5 — Target list for this run
------------------------------------------------------------------------
  Registry total   : 25
  Gated / too big  : 8
  Target this run  : 17
    - google-bert/bert-base-uncased                            5/6 done
    - distilbert/distilbert-base-uncased                       0/6 done
    - FacebookAI/roberta-base                                  0/6 done
    - google/electra-small-discriminator                       0/6 done
    - microsoft/deberta-v3-small                               0/6 done
    - gpt2                                                     0/6 done
    - EleutherAI/gpt-neo-125m                                  0/6 done
    - facebook/opt-125m                                        0/6 done
    - HuggingFaceTB/SmolLM2-135M                               0/6 done
    - HuggingFaceTB/SmolLM2-360M                               0/6 done
    - Qwen/Qwen2-0.5B   

## Pre-flight — the go/no-go gate

Everything below happens **before** a single model is loaded. It is the last chance to abort before a multi-hour run.

### The five checks

1. **Slug collisions.** Two models whose names differ only by org map to the same output folder and would silently overwrite each other.
2. **HF cache writable.** If the cache directory exists but is not writable, model downloads fail two minutes into the first model load.
3. **Order manifest valid JSON.** `extractio_order.json` sorts the run. Malformed files are silently ignored by `Extraction.py`; we want a loud failure.
4. **RAM sanity.** For each target model, estimate the fp32 load peak. Warn if peak > 80% of available RAM.
5. **Disk space.** Warn if free disk is under 2× the estimated output size.

Hard failures (1, 2, 3, 5) raise `RuntimeError`. RAM emits warnings only.


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8 — PRE-FLIGHT CHECKS
#
# All checks are ADVISORY except two hard-blockers:
#   - slug collisions (would silently overwrite artifacts)
#   - unwritable HF cache (would fail two minutes into the first load)
# Everything else — RAM, disk, manifest — prints a warning and lets the
# user decide. Extraction.py has its own per-model RAM guard; the disk
# estimate is a rough floor that assumes the largest dataset on the
# largest model.
# ═══════════════════════════════════════════════════════════════════════

_banner("STEP 4 / 5 — Pre-flight checks")

# Per-model per-sample hidden-state cost, in bytes.
# 25 layers × 896 hidden × 4 bytes fp32 ≈ 89,600 B for Qwen-class.
# Larger models cost more, but this is the floor for the smaller ones.
def _bytes_per_sample(model_name):
    spec = EX.get_model_spec(model_name)
    # Try to read the actual config from the materialized snapshot if present.
    try:
        cfg_path = EX.build_materialized_snapshot_dir(model_name) / "config.json"
        if cfg_path.is_file():
            cfg = json.loads(cfg_path.read_text())
            layers = (cfg.get("num_hidden_layers")
                      or cfg.get("n_layer")
                      or cfg.get("num_layers"))
            hidden = (cfg.get("hidden_size")
                      or cfg.get("d_model")
                      or cfg.get("n_embd"))
            if layers and hidden:
                return (layers + 1) * hidden * 4
    except Exception:
        pass
    # Fallback: worst-case 25 layers × 896 hidden.
    return 25 * 896 * 4

def _rough_peak_gb(billions):
    """fp32 weights × 2 (transient load double) + 0.5 GB activation."""
    return billions * 4.0 * 2 + 0.5

_errors   = []
_warnings = []

# ── 1. Slug collisions — HARD ─────────────────────────────────────────
_slug_map = {}
for m in ALL_MODELS:
    _slug_map.setdefault(model_slug(m), []).append(m)
_collisions = {s: ms for s, ms in _slug_map.items() if len(ms) > 1}
if _collisions:
    for s, ms in _collisions.items():
        _errors.append(f"Slug collision '{s}': {ms}")
    _say("  X  slug collisions detected")
else:
    _say("  OK no slug collisions")

# ── 2. HF cache writable — HARD ───────────────────────────────────────
if HF_HUB_CACHE.exists() and not os.access(HF_HUB_CACHE, os.W_OK):
    _errors.append(f"HF cache not writable: {HF_HUB_CACHE}")
    _say("  X  HF cache not writable")
else:
    _say(f"  OK HF cache writable")

# ── 3. Order manifest — SOFT ──────────────────────────────────────────
_ORDER_CANDIDATES = [
    Path("extractio_order.json"),
    Path("extraction_order.json"),
    EXTRACTION_META_ROOT / "extraction_order.json",
]
_order_file = next((p for p in _ORDER_CANDIDATES if p.is_file()), None)
if _order_file is None:
    _warnings.append("No extraction_order.json — default registry order used")
    _say("  -- no order manifest (using default registry order)")
else:
    try:
        _order = json.loads(_order_file.read_text())
        _say(f"  OK order manifest valid: {_order_file.name} "
             f"({len(_order.get('model_order', []))} models, "
             f"{len(_order.get('dataset_order', []))} datasets)")
    except Exception as exc:
        _errors.append(f"Order manifest invalid: {_order_file} — {exc}")
        _say("  X  order manifest invalid")

# ── 4. RAM sanity — SOFT ──────────────────────────────────────────────
# Extraction.py has its own preflight: it estimates the loaded weight
# size and refuses to load a model whose peak exceeds the machine's
# safe total. So this check is a heads-up, not a gate.
try:
    import psutil
    _vm = psutil.virtual_memory()
    _avail_gb = _vm.available / 1e9
    _say(f"  -- available RAM: {_avail_gb:.2f} GB")

    _ram_warn = []
    for m in TARGET_MODELS:
        spec = EX.get_model_spec(m)
        peak = _rough_peak_gb(spec.parameter_billions)
        if peak > _avail_gb * 0.8:
            _ram_warn.append((m, peak))

    if _ram_warn:
        _say(f"  !  {len(_ram_warn)} model(s) may exceed available RAM:")
        for m, p in _ram_warn:
            _say(f"       {m:<55}  peak ~{p:.1f} GB")
        _say("     Extraction.py will skip models that fail its own check.")
    else:
        _say("  OK all target models fit in RAM")
except ImportError:
    _say("  -- psutil not installed — RAM check skipped")

# ── 5. Disk space — SOFT, with per-dataset breakdown ──────────────────
import shutil
_free_gb = shutil.disk_usage(str(HIDDEN_STATES_ROOT.parent)).free / 1e9

_est_by_dataset = {}
for d in DS_NAMES:
    n_pending_models = sum(
        1 for m in TARGET_MODELS if not dataset_is_done(m, d)
    )
    if n_pending_models == 0:
        continue
    # Average bytes/sample across the pending models.
    avg_bps = sum(_bytes_per_sample(m) for m in TARGET_MODELS) / max(1, len(TARGET_MODELS))
    _est_by_dataset[d] = (len(DATASETS[d]) * avg_bps * n_pending_models) / 1e9

_est_output_gb = sum(_est_by_dataset.values())

_say(f"  -- free disk: {_free_gb:.1f} GB  |  est. output: {_est_output_gb:.1f} GB")

# Show the top contributors so the user can decide what to drop.
_top = sorted(_est_by_dataset.items(), key=lambda x: -x[1])[:5]
if _top:
    _say("  -- largest contributors:")
    for name, gb in _top:
        _say(f"       {name:<24}  {gb:>7.1f} GB")

if _free_gb < _est_output_gb:
    _warnings.append(
        f"Disk estimate ({_est_output_gb:.0f} GB) exceeds free space "
        f"({_free_gb:.0f} GB). The run will proceed; extraction will fail "
        f"when the disk fills. Consider excluding large datasets."
    )
    _say("  !  disk will likely fill during the run")
elif _free_gb < _est_output_gb * 2:
    _warnings.append(
        f"Disk headroom is tight: {_free_gb:.0f} GB free vs {_est_output_gb:.0f} GB estimated."
    )
    _say("  !  tight disk headroom")
else:
    _say("  OK sufficient disk")

# ── Verdict ───────────────────────────────────────────────────────────
print()
if _errors:
    for e in _errors:
        print(f"  X {e}")
    raise RuntimeError(
        "Pre-flight failed on hard-blocker checks. Fix the issues above "
        "and re-run."
    )

if _warnings:
    print()
    for w in _warnings:
        print(f"  ! {w}")
    display_info(
        f"<b>{len(_warnings)} warning(s).</b> The run will proceed. "
        f"Extraction.py's own per-model RAM guard will skip models that "
        f"exceed memory. Monitor disk usage during the run."
    )
else:
    display_info("<b>Pre-flight passed.</b> Next cell launches the extraction run.")


------------------------------------------------------------------------
STEP 4 / 5 — Pre-flight checks
------------------------------------------------------------------------
  OK no slug collisions
  OK HF cache writable
  OK order manifest valid: extractio_order.json (25 models, 6 datasets)
  -- available RAM: 2.38 GB
  !  8 model(s) may exceed available RAM:
       HuggingFaceTB/SmolLM2-360M                               peak ~3.4 GB
       Qwen/Qwen2-0.5B                                          peak ~4.5 GB
       Qwen/Qwen2.5-0.5B                                        peak ~4.5 GB
       Qwen/Qwen3-0.6B-Base                                     peak ~5.3 GB
       Qwen/Qwen2-1.5B                                          peak ~12.5 GB
       Qwen/Qwen2.5-1.5B                                        peak ~12.8 GB
       HuggingFaceTB/SmolLM2-1.7B                               peak ~14.1 GB
       TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T      peak ~9.3 GB
     Extractio

## Launch — the extraction run

The next cell calls `Extraction.run_experiments` with:

| Parameter | Value | Meaning |
|---|---|---|
| `datasets` | from cell 4 | the DataFrames to tokenize |
| `model_names` | `TARGET_MODELS` | from cell 6 |
| `pooling` | `"mean"` | average hidden state over all tokens |
| `max_length` | `128` | truncation length |
| `use_half_precision` | `True` | fp16 forward, fp32 storage |
| `flush_every_batches` | `8` | `fsync` cadence |
| `continue_on_model_error` | `True` | log and skip on failure |
| `experiment_id` | `"master_v2"` | tag in every artifact's metadata |

### Timing

A 0.5B model takes 10–20 minutes per dataset on CPU. Full 18-model × 6-dataset matrix: 4–8 hours. Run in `tmux` or `nohup` so a Wi-Fi blip does not kill the job.


In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 10 — LAUNCH
# ═══════════════════════════════════════════════════════════════════════

_banner("STEP 5 / 5 — Extraction run")

if not TARGET_MODELS:
    display_info("No models to extract.")
else:
    _say(f"  Models   : {len(TARGET_MODELS)}")
    _say(f"  Datasets : {len(DATASETS)}")
    _say(f"  Output   : {HIDDEN_STATES_ROOT}")
    _say("")

    _t0 = time.perf_counter()
    results = EX.run_experiments(
        datasets=DATASETS_FILTERED,
        dataset_csv_hashes=CSV_HASHES_FILTERED,
        model_names=TARGET_MODELS,
        pooling="mean",
        max_length=128,
        use_half_precision=True,
        flush_every_batches=8,
        continue_on_model_error=True,
        experiment_id="master_v2",
        show_verbose=(VERBOSITY == "debug"),
        show_info=(VERBOSITY != "quiet"),
        show_critical=True,
        show_debug=(VERBOSITY == "debug"),
    )
    _elapsed = time.perf_counter() - _t0

    _banner("STEP 5 / 5 — Extraction finished")
    _say(f"  Elapsed  : {_elapsed/60:.1f} min")
    _say(f"  Records  : {len(results) if results else 0}")
    print("\nNext cell summarises the run.")



------------------------------------------------------------------------
STEP 5 / 5 — Extraction run
------------------------------------------------------------------------
  Models   : 17
  Datasets : 6
  Output   : /Volumes/Amirali/hidden_states

  ✓ HF cache configured: /Volumes/Amirali/.hf_cache/hub

↕ Extraction order loaded from /Users/amirali/Desktop/Final Year Project/Final-Year-Project/extractio_order.json
    models   : 17
    datasets : 5
    ⚠ 8 model entries not in this run:
        · Qwen/Qwen3-1.7B-Base
        · Qwen/Qwen2.5-3B
        · Qwen/Qwen3-4B-Base
        · google/gemma-3-270m
        · google/gemma-3-1b-pt
        · … and 3 more
    ⚠ dataset entries not discovered: ['amazon_polarity']

╔══════════════════════════════════════════════════════════════════════════════════════╗
║ HIDDEN STATE EXTRACTION EXPERIMENT                                                   ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
  Experim

`torch_dtype` is deprecated! Use `dtype` instead!


  ✓ Tokenizer loaded in 1.26s Mode=fast

MODEL LOAD ATTEMPT
  Attempt              : 1/2
  Path                 : primary
  Model                : Qwen/Qwen2-0.5B
  Snapshot             : /Volumes/Amirali/models/Qwen2-0.5B
  dtype                : torch.float32
  device               : cpu
  SDPA                 : False
  low_cpu_mem_usage    : True
  LOADING RESULT
    Missing keys        : 0
    Non-core missing    : 0
    Core missing        : 0
    Unexpected keys     : 0
    Mismatched keys     : 0
    Meta tensors        : 0
    Core meta           : 0
    Fatal discrepancy   : False

✓ MODEL READY
  Load/probe time       : 144.01s
  Backend               : model_default
  Model device          : cpu
  Hidden states         : 25
  Hidden size           : 896

────────────────────────────────────────────────────────────────────────────────────────
DATASET isear
────────────────────────────────────────────────────────────────────────────────────────
  Text column         : clean_te

emotion:  15%|#4        | 2939/19999 [00:00<?, ?sample/s]

[start] starting with 17060 incomplete samples

────────────────────────────────────────────────────────────────────────────────────────
RUNTIME / SPEED TELEMETRY :: emotion
────────────────────────────────────────────────────────────────────────────────────────
  Progress             : 2,940/19,999 (14.70%)
  Newly computed       : 1
  Wall time            : 53.07s
  Throughput           : 0.02 samples/s
  ETA                  : 251h 27m 46.9s
  Current stage        : measurement

  LAST BATCH
    Range              : 2939:2940
    New samples        : 1
    Total              : 51.70s
    Forward            : 50.77s
    Tokenization       : 0.33s
    Input transfer     : 0.00s
    Pooling            : 0.18s
    Conversion         : 0.03s
    Memmap write       : 0.14s
    Flush              : 0.00s
    Sequence length    : 12
    Tokens             : 12
    Token throughput   : 0.23 tokens/s

  ROLLING PERFORMANCE
    Mean batch         : 51.697s
    Median batch       : 51.697s
    

## Post-run summary

The next cell rebuilds the completion matrix from disk, tail-prints the ledger at `hidden_states/_meta/ledger.jsonl`, and charts cumulative extraction time per model.

If a row is still partial, the model either failed or was interrupted. Re-run cell 10 to resume — the pipeline picks up from the last completed sample. The ledger records every (model, dataset) run with timing and outcome, so you can compute per-model throughput after the fact.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 12 — POST-RUN SUMMARY
# ═══════════════════════════════════════════════════════════════════════

_banner("POST-RUN — Completion matrix")

post_matrix = pd.DataFrame(
    [[("V" if dataset_is_done(m, d) else ".") for d in DS_NAMES]
     for m in TARGET_MODELS],
    index=[m.split("/")[-1] for m in TARGET_MODELS],
    columns=DS_NAMES,
)
post_matrix["done"] = (post_matrix[DS_NAMES] == "V").sum(axis=1).astype(str) + f"/{len(DS_NAMES)}"
display(post_matrix)

_ledger = EXTRACTION_META_ROOT / "ledger.jsonl"
if _ledger.is_file():
    _banner("POST-RUN — Recent ledger entries")
    for line in _ledger.read_text().strip().splitlines()[-5:]:
        try:
            r = json.loads(line)
            model   = r.get("model", {}).get("name", "?")
            dataset = r.get("dataset", {}).get("name", "?")
            status  = r.get("status", "?")
            print(f"  {model:<45} {dataset:<22} {status}")
        except Exception:
            continue

_rows = []
for m in TARGET_MODELS:
    for d in DS_NAMES:
        p = EX.build_dataset_directory(m, d) / "extraction.json"
        if not p.is_file():
            continue
        try:
            meta = json.loads(p.read_text())
            secs = meta.get("performance", {}).get("elapsed_seconds")
            if secs:
                _rows.append({"model": m.split("/")[-1], "dataset": d, "elapsed_s": secs})
        except Exception:
            continue

if _rows:
    _df_time = (pd.DataFrame(_rows)
                  .groupby("model")["elapsed_s"].sum()
                  .sort_values(ascending=False)
                  .reset_index())

    sns.set_style("darkgrid")
    plt.rcParams.update({
        "figure.facecolor": "#1e1e1e", "axes.facecolor": "#2d2d2d",
        "axes.edgecolor": "#d4d4d4",   "axes.labelcolor": "#d4d4d4",
        "text.color": "#d4d4d4",        "xtick.color": "#d4d4d4",
        "ytick.color": "#d4d4d4",       "grid.color": "#444444",
    })
    fig, ax = plt.subplots(figsize=(12, max(3, 0.4 * len(_df_time))))
    sns.barplot(data=_df_time, x="elapsed_s", y="model", palette="magma", ax=ax)
    ax.set_xlabel("Cumulative extraction time (seconds)")
    ax.set_title("Extraction time per model (all datasets)", color="#4fc3f7")
    plt.tight_layout()
    plt.show()

    _out = Path("extraction_summary.csv")
    pd.DataFrame(_rows).to_csv(_out, index=False)
    display_info(f"Summary exported to <code>{_out.resolve()}</code>")
else:
    display_info("No extraction metadata found yet.")
